# Chunking

**Goal:** Apply chunking strategies to a real messy corpus and see how they change retrieval.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


In [ ]:
%pip install -q anthropic requests

In [ ]:
import os

# In Colab, read the key from Secrets (key icon in the left sidebar).
# Locally, set the ANTHROPIC_API_KEY env var instead.
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except ImportError:
    assert os.environ.get('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY'

import anthropic
client = anthropic.Anthropic()
MODEL = 'claude-sonnet-5'  # good default: capable and cheap enough to iterate on

## The corpus: ten IETF RFCs

Every RAG system starts with a corpus, and this repo uses one corpus for the whole RAG and evals arc: ten IETF RFCs — IP, TCP, DNS, HTTP/1.1, BGP, SMTP, WebSocket, OAuth 2.0, HTTP/2, and HTTP Semantics. The evals in section 03 will measure retrieval over the exact chunks you build here.

RFCs are a good teaching corpus precisely because they're messy in the ways real corpora are messy: page headers and footers every 58 lines, form-feed characters between pages, ASCII diagrams, and section conventions that drifted over 40 years of publication. If your pipeline handles these cleanly, a pile of internal wiki exports won't scare you.


In [ ]:
# Download the shared corpus: ten IETF RFCs, cached to data/rfc/.
# Every notebook in this repo that needs the corpus includes this cell --
# self-containment over DRY, so each notebook runs top-to-bottom on its own.
import os
import requests

RFC_NUMBERS = [791, 793, 1035, 2616, 4271, 5321, 6455, 6749, 7540, 9110]
RFC_TITLES = {
    791: 'IP', 793: 'TCP', 1035: 'DNS', 2616: 'HTTP/1.1', 4271: 'BGP',
    5321: 'SMTP', 6455: 'WebSocket', 6749: 'OAuth 2.0', 7540: 'HTTP/2',
    9110: 'HTTP Semantics',
}
DATA_DIR = 'data/rfc'
os.makedirs(DATA_DIR, exist_ok=True)

corpus = {}  # rfc number -> raw text
for num in RFC_NUMBERS:
    path = os.path.join(DATA_DIR, f'rfc{num}.txt')
    if not os.path.exists(path):  # cached: skip the download on re-runs
        resp = requests.get(f'https://www.rfc-editor.org/rfc/rfc{num}.txt', timeout=30)
        resp.raise_for_status()
        with open(path, 'w') as f:
            f.write(resp.text)
    with open(path) as f:
        corpus[num] = f.read()

for num in RFC_NUMBERS:
    print(f'RFC {num:>4}  {RFC_TITLES[num]:<15} {len(corpus[num]):>9,} chars')

## Clean before you chunk

Open RFC 2616 and you'll find page furniture: a footer line like `Fielding, et al.  Standards Track  [Page 5]`, a form-feed character, then a header line like `RFC 2616  HTTP/1.1  June 1999` — repeated every 58 lines. Left in place, that junk lands in the middle of your chunks and gets embedded as if it were content.

And the corpus isn't even consistently messy: RFC 9110 (2022) is published in the modern format with no pagination at all, while the 1981 RFCs use a *different* header style (`September 1981` on its own line) that the regexes below miss. That mix — several generations of formatting conventions in one corpus — is exactly what real document collections look like.

The cleaner below is three regexes. It doesn't need to be smarter than that — but it does need to exist. Skipping this step is the most common self-inflicted wound in RAG pipelines: garbage in the chunks means garbage in the embeddings, and no retrieval trick downstream repairs it.


In [ ]:
import re

def clean_rfc(text):
    """Strip RFC page furniture: form feeds, page footers, page headers."""
    text = text.replace('\f', '\n')
    lines = []
    for line in text.split('\n'):
        if re.match(r'^.*\[Page \d+\]\s*$', line):        # 'Author, et al. ... [Page 12]'
            continue
        if re.match(r'^RFC \d+\s+.*\S+ \d{4}\s*$', line):  # 'RFC 9110  HTTP Semantics  June 2022'
            continue
        lines.append(line)
    text = '\n'.join(lines)
    return re.sub(r'\n{3,}', '\n\n', text).strip()  # collapse the gaps the furniture left

cleaned = {num: clean_rfc(corpus[num]) for num in RFC_NUMBERS}

In [ ]:
# Before/after: the same page break in RFC 2616, raw vs cleaned.
raw = corpus[2616]
i = raw.find('[Page 20]')   # a break in the middle of body text
print('--- RAW (around the page-20 break) ---')
print(repr(raw[i - 150:i + 250]))

# Grab the last real content line before the footer and find it in the
# cleaned text, so we can show the same region with the furniture gone.
before = [l for l in raw[:i].split('\n') if len(l.strip()) > 20][-1]
j = cleaned[2616].find(before.strip()[:50])
print()
print('--- CLEANED (same region) ---')
print(repr(cleaned[2616][j:j + 300]))

Run this and note what disappeared: the footer, the `\f` form feed, and the header — and that text cut in half by the page break now reads continuously. That continuity matters for every chunker below.

One honesty note before moving on: run `[l for l in cleaned[793].split('\n') if l.strip() == 'September 1981'][:3]` and you'll see the 1981-style headers leaked through. Cleaning is never *done* — you decide when it's good enough, and for this corpus, good enough is fine.

## Three chunkers, from scratch

No libraries — the three strategies below are about 30 lines total, and writing them yourself makes the tradeoffs concrete.

1. **Fixed-size with overlap** — slice every `size` characters, overlapping by `overlap`. Knows nothing about the text. The overlap is a hedge: if a boundary cuts an answer in half, maybe the neighboring chunk still holds the whole thing.
2. **Paragraph-based** — split on blank lines, then pack paragraphs into chunks up to a budget. Respects the smallest unit of structure the author gave you.
3. **Section-aware** — split on RFC section headings (`5.3.  Ordering of Field Lines`). This works because RFC body text is indented and headings sit at column zero — one regex captures the document's real structure. Long sections fall back to paragraph packing, carrying the heading along so each piece keeps its context.


In [ ]:
def chunk_fixed(text, size=1200, overlap=200):
    chunks, step = [], size - overlap
    for start in range(0, len(text), step):
        piece = text[start:start + size]
        if piece.strip():
            chunks.append(piece)
        if start + size >= len(text):
            break
    return chunks

def chunk_paragraphs(text, max_chars=1200):
    paras = [p for p in text.split('\n\n') if p.strip()]
    chunks, cur = [], ''
    for p in paras:
        if cur and len(cur) + len(p) + 2 > max_chars:
            chunks.append(cur)
            cur = p
        else:
            cur = cur + '\n\n' + p if cur else p
    if cur:
        chunks.append(cur)
    return chunks

HEADING_RE = re.compile(r'^\d+(?:\.\d+)*\.?\s+\S', re.MULTILINE)

def chunk_sections(text, max_chars=2400):
    starts = [m.start() for m in HEADING_RE.finditer(text)]
    if not starts:
        return chunk_paragraphs(text, max_chars)
    chunks = [text[:starts[0]].strip()] if text[:starts[0]].strip() else []
    for a, b in zip(starts, starts[1:] + [len(text)]):
        sec = text[a:b].strip()
        if len(sec) <= max_chars:
            chunks.append(sec)
        else:  # long section: paragraph-pack it, carrying the heading along
            heading = sec.split('\n', 1)[0]
            for sub in chunk_paragraphs(sec, max_chars):
                chunks.append(sub if sub.startswith(heading) else heading + '\n' + sub)
    return [c for c in chunks if c]

## Compare them: counts and size distributions

Chunk size is a retrieval-precision vs context-cost tradeoff. Small chunks are precise — a hit is mostly answer, not padding — but risk splitting answers apart, and you need more of them in the prompt to cover a topic. Big chunks keep answers intact but each retrieved hit drags along more irrelevant text, which costs tokens and dilutes the model's attention.

Note the section-aware distribution: it's the *widest*, because sections are as long as they are. That's not a defect — it means chunk boundaries land where the author put topic boundaries, instead of where a character counter happened to be.


In [ ]:
import statistics

strategies = {
    'fixed (1200/200)': chunk_fixed,
    'paragraph (1200)': chunk_paragraphs,
    'section (2400)':   chunk_sections,
}

all_chunks = {}
for name, fn in strategies.items():
    chunks = []
    for num in RFC_NUMBERS:
        chunks.extend(fn(cleaned[num]))
    all_chunks[name] = chunks

print(f'{"strategy":<18} {"chunks":>7} {"min":>6} {"median":>7} {"max":>7}')
for name, chunks in all_chunks.items():
    sizes = [len(c) for c in chunks]
    print(f'{name:<18} {len(chunks):>7} {min(sizes):>6} '
          f'{int(statistics.median(sizes)):>7} {max(sizes):>7}')

## The load-bearing demo: one question, three boundary behaviors

Take a concrete question: *"What status code does HTTP use when the request body is too large?"* The answer lives in RFC 9110, section 15.5.14 — **413 Content Too Large** — a heading followed by two short paragraphs.

For each strategy, find the chunks that mention it and look at where they start and end. This is the whole argument about chunking in one printout.


In [ ]:
PHRASE = 'Content Too Large'   # the RFC 9110 name for status code 413

for name, chunks in all_chunks.items():
    hits = [i for i, c in enumerate(chunks) if PHRASE in c]
    print(f'--- {name}: {len(hits)} chunk(s) mention {PHRASE!r}')
    for i in hits[:3]:
        c = chunks[i]
        print(f'  [chunk {i}, {len(c)} chars]')
        print(f'    starts: {c[:70]!r}')
        print(f'    ends:   {c[-70:]!r}')
    print()

Run this and note three things:

- **Fixed-size** chunks containing the phrase typically start and end mid-sentence — the character counter doesn't care that section 15.5.14 exists. Whether the heading `413 Content Too Large` and the sentence explaining *when* a server sends it land in the same chunk is luck. The 200-character overlap sometimes rescues it — that's what overlap is for — but it's a hedge, not a fix: it narrows the window in which a split hurts, it doesn't remove it.
- **Paragraph-based** chunks end at paragraph boundaries, so sentences survive — but the heading and its body are separate paragraphs, and the packer may put them in different chunks. A chunk that says "the server is unwilling to process a payload this large" without the string `413` above it is a chunk that keyword search can never connect to the question.
- **Section-aware** produces exactly one chunk that starts at `15.5.14.  413 Content Too Large` and contains the whole section — heading, code, and explanation travel together. (The other hits are table-of-contents lines that merely list the phrase; retrieval will rank the real section above them because it actually discusses the topic.)

The general lesson: **structure-aware beats clever-generic when the corpus has structure** — and most corpora do. RFCs have numbered sections; your docs have Markdown headings; contracts have clauses; code has functions. Spend your effort finding the structure before you spend it tuning chunk sizes.

One caveat to carry forward: the section chunker is only as good as its heading regex. A corpus with inconsistent headings (and RFC 791, from 1981, has some) will leak mis-splits. Check a sample of chunks by eye before trusting any chunker — that habit is cheaper than any downstream fix.


## Exercises

1. **Token-based fixed chunking.** Rewrite `chunk_fixed` to measure size in tokens instead of characters (use `len(text.split())` as a cheap proxy, or a real tokenizer). Re-run the comparison table — does the size distribution get tighter? Why do production systems budget in tokens?
2. **Overlap sweep.** Run `chunk_fixed` with overlap 0, 100, 200, and 400 over RFC 9110 and count, for each setting, how many of the chunks containing `'413 (Content Too Large)'` also contain the phrase `'willing or able to process'` (the body of the section). Plot or print the fraction — you're measuring how much the hedge actually buys.
3. **Heading-path metadata.** Extend `chunk_sections` to attach the section number and title (e.g. `15.5.14. 413 Content Too Large`) as metadata on each chunk instead of just leaving it in the text. Print five random chunks with their metadata. In notebook 02 this becomes citation material.
4. **Break the section chunker.** Find a place in RFC 791 or RFC 793 (the two oldest) where `HEADING_RE` mis-splits — either a heading it misses or a non-heading line it fires on. Fix the regex without breaking it for RFC 9110, and note how much fiddlier this gets as the corpus gets older.
